In [1]:
import xarray as xr
import numpy as np

In [2]:
# 1. Open the dataset with anonymous GCS access
total_ds = xr.open_zarr(
    "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3",
    chunks=None,
    storage_options=dict(token="anon")
)

print(list(total_ds.data_vars))

['100m_u_component_of_wind', '100m_v_component_of_wind', '10m_u_component_of_neutral_wind', '10m_u_component_of_wind', '10m_v_component_of_neutral_wind', '10m_v_component_of_wind', '10m_wind_gust_since_previous_post_processing', '2m_dewpoint_temperature', '2m_temperature', 'air_density_over_the_oceans', 'angle_of_sub_gridscale_orography', 'anisotropy_of_sub_gridscale_orography', 'benjamin_feir_index', 'boundary_layer_dissipation', 'boundary_layer_height', 'charnock', 'clear_sky_direct_solar_radiation_at_surface', 'cloud_base_height', 'coefficient_of_drag_with_waves', 'convective_available_potential_energy', 'convective_inhibition', 'convective_precipitation', 'convective_rain_rate', 'convective_snowfall', 'convective_snowfall_rate_water_equivalent', 'downward_uv_radiation_at_the_surface', 'duct_base_height', 'eastward_gravity_wave_surface_stress', 'eastward_turbulent_surface_stress', 'evaporation', 'forecast_albedo', 'forecast_logarithm_of_surface_roughness_for_heat', 'forecast_surface

In [3]:
variable_list = ["2m_temperature", "100m_u_component_of_wind", '100m_v_component_of_wind', 'total_precipitation',]
year_list = list(range(2025, 2014, -1))
month_list = list(range(1, 13))
print(year_list)

[2025, 2024, 2023, 2022, 2021, 2020, 2019, 2018, 2017, 2016, 2015]


In [4]:
import os

root_folder = "D:\Download"     #-------------------------------------- Customize for your OS.
def get_file_path(year, month, day, variable):
    return os.path.join(root_folder, str(year), str(month).zfill(2), str(day).zfill(2), variable + ".npz")

In [5]:
import calendar
def get_last_day_of_month(year, month):
    return calendar.monthrange(year, month)[1]

In [ ]:
# 2. Select variables and time range
import datetime
# lastday = datetime.today() - datetime.timedelta(days=6)
lastday = datetime.date(2025,7,31) - datetime.timedelta(days=6)

for year in year_list:
    for month in month_list:
        for day in range(1, get_last_day_of_month(year, month)+1):
            if datetime.date(year, month, day) > lastday: continue
            for variable in variable_list:
                print("{}/{}/{} {}...".format(year, str(month).zfill(2), str(day).zfill(2), variable), end='')

                file_path = get_file_path(year, month, day, variable)
                if os.path.exists(file_path):
                    print(''); continue

                ds = total_ds[[variable]]
                monthDayDtr = str(year) + '-' + str(month).zfill(2) + '-' + str(day).zfill(2)
                ds = ds.sel(time=slice(monthDayDtr, monthDayDtr))

                # 3. Extract numpy arrays
                array = ds[variable].values
                array = array.astype(np.float16)
                print(" {}".format(array.shape))

                # 4. Save to disk as raw binary numpy files
                os.makedirs(os.path.dirname(file_path), exist_ok=True)
                np.savez_compressed(file_path, array)

2025/01/01 2m_temperature...
2025/01/01 100m_u_component_of_wind... (24, 721, 1440)
2025/01/01 100m_v_component_of_wind... (24, 721, 1440)
2025/01/01 total_precipitation...
2025/01/02 2m_temperature...
2025/01/02 100m_u_component_of_wind... (24, 721, 1440)
2025/01/02 100m_v_component_of_wind... (24, 721, 1440)
2025/01/02 total_precipitation...
2025/01/03 2m_temperature...
2025/01/03 100m_u_component_of_wind... (24, 721, 1440)
2025/01/03 100m_v_component_of_wind... (24, 721, 1440)
2025/01/03 total_precipitation...
2025/01/04 2m_temperature...
2025/01/04 100m_u_component_of_wind... (24, 721, 1440)
2025/01/04 100m_v_component_of_wind... (24, 721, 1440)
2025/01/04 total_precipitation...
2025/01/05 2m_temperature...
2025/01/05 100m_u_component_of_wind... (24, 721, 1440)
2025/01/05 100m_v_component_of_wind... (24, 721, 1440)
2025/01/05 total_precipitation...
2025/01/06 2m_temperature...
2025/01/06 100m_u_component_of_wind... (24, 721, 1440)
2025/01/06 100m_v_component_of_wind... (24, 721, 14